In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import pyarrow as pa
import pyarrow.parquet as pq
import time
from fake_useragent import UserAgent
import random
import json
from pprint import pprint
import csv
import os

In [ ]:
session=requests.Session()
url='https://irr.by/cars/passenger'
ua=UserAgent()
links_auto=[]
CSV_FILE='auto_data.csv'

In [ ]:
def get_links(url):
    page=1
    while True:
        headers={'User-Agent':ua.random}
        params={'page':page}

        response=session.get(url, headers=headers, params=params)

        if response.status_code!=200:
            break

        if 'Извините, но объявлений по Вашему запросу не найдено.' in response.text:
            print(f'страница {page-1} была последняя')
            break

        soup=BeautifulSoup(response.text, "html.parser")

        div_cards=soup.find_all('div', class_=re.compile('ListingWrapperPage-module__'))

        if not div_cards:
            print('карточки авто не найдены')
            break

        for div in div_cards:
            clink=div.find('a')

            if clink and clink.has_attr('href'):
                href=clink['href']
                if href.startswith('/'):
                    continue
                links_auto.append(href)

        sleep=random.uniform(1, 4)
        print(f'sleep {sleep:.2f}s')
        time.sleep(sleep)
        page+=1

get_links(url)
print(links_auto)

In [ ]:
print(len(links_auto))
print(links_auto)

In [ ]:
with open('links_auto.txt', 'w') as f:
    for link in links_auto:
        f.write(link+'\n')

In [ ]:
with open('links_auto.txt', 'r') as f:
    links_auto=f.read().splitlines()

In [ ]:
def safe_int(value, default=0):
    try:
        return int(value) if value is not None else default
    except(ValueError, TypeError):
        return default

def safe_float(value, default=0):
    try:
        return float(value) if value is not None else default
    except(ValueError, TypeError):
        return default

def get_data(url):
    headers = {'User-Agent': ua.random}

    max_retries=3
    retry_delay=60
    for i in range(1, max_retries+1):
        try:
            response=session.get(url, headers=headers, timeout=15)
            break
        except requests.exceptions.RequestException as e:
            print(f"сбой доступа, попытка {i}/{max_retries} не удалась, ошибка {e}")
            if i<max_retries:
                print("ожидаем минуту")
                time.sleep(retry_delay)
            else:
                print(f'{max_retries} попытки исчерпаны, пропускаем ссылку {url}')
                return None
    
    
    if response.status_code!=200:
        return None
    
    soup=BeautifulSoup(response.text, 'html.parser')
    
    json_data=soup.find('script', type='application/ld+json')
    
    if json_data:
        data=json.loads(json_data.string)
        
        car_info=data[0] if isinstance(data, list) else data
        
        title=car_info.get('name')
        price=car_info.get('offers', {}).get('price')
        currency=car_info.get('offers', {}).get('priceCurrency')
        description=car_info.get('description')
        
        specs={}
        
        for prop in car_info.get('additionalProperty', []):
            specs[prop['name']]= prop['value']
            
        car_data = {
            'name': title,
            'price': safe_float(price),
            'currency': currency,
            'brand': specs.get('Марка'),
            'model': specs.get('Модель'),
            'year': safe_int(specs.get('Год выпуска')),
            'mileage': safe_float(specs.get('Пробег')),
            'body_type': specs.get('Тип кузова'),
            'engine_volume': safe_float(specs.get('Объем двигателя')),
            'fuel_type': specs.get('Тип двигателя'),
            'transmission': specs.get('Тип трансмиссии'),
            'color': specs.get('Цвет'),
            'description': description,
            'link': url
        }
        return car_data
    else:
        print('no json')
        return None

In [ ]:
FIELDS=['name', 'price', 'currency', 'brand', 'model', 'year', 'mileage', 'body_type', 'engine_volume', 'fuel_type', 'transmission', 'color', 'description', 'link']

def save_csv(car_data, filename=CSV_FILE):
    if car_data is None:
        return
    filen=os.path.isfile(filename)

    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f, fieldnames=FIELDS, delimiter=';')

        if not filen:
            writer.writeheader()

        writer.writerow(car_data)

In [ ]:
for link in links_auto:
    save_csv(get_data(link))
    sleep=random.uniform(2, 5)
    print(f'sleep {sleep:.2f}s')
    time.sleep(sleep)

In [ ]:
df=pd.read_csv('auto_data.csv', sep=';', encoding='utf-8-sig')

if 'description' in df.columns:
    df['description']=df['description'].astype(str).str.replace(r'[\r\n]+', ' ', regex=True)

df.to_csv('auto_data.csv', sep=';', index=False, encoding='utf-8-sig')

In [ ]:
df=pd.read_csv('auto_data.csv', sep=';', encoding='utf-8-sig')
schema=pa.schema([
    ('name', pa.string()),
    ('price', pa.int64()),
    ('currency', pa.string()),
    ('brand', pa.string()),
    ('model', pa.string()),
    ('year', pa.int64()),
    ('mileage', pa.int64()),
    ('body_type', pa.string()),
    ('engine_volume', pa.float64()),
    ('fuel_type', pa.string()),
    ('transmission', pa.string()),
    ('color', pa.string()),
    ('description', pa.string()),
    ('link', pa.string())
])

table=pa.Table.from_pandas(df, schema=schema)
pq.write_table(table, 'auto.parquet')

In [ ]:
tabler = pq.read_table("auto.parquet")
dfr = tabler.to_pandas()

# --- Output example ---

# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_colwidth', None)

# display(dfr)

# ----------------------